# Click registration landmarks (FaceRhythm, cameras without LP: cam1 / cam2 / cam3)

For `scripts/face_rhythm_register.py --cam <CAM>`. Run in the **`fr`** env in **JupyterLab** (like `face_rhythm_draw_roi.ipynb`).

1. First: `python scripts/face_rhythm_register.py --cam <CAM> --stage templates` -- picks each animal's template session and the master and writes `register_inputs/click_<session>.png` + `click_sessions.json`.
2. Here: for each listed session, **click the landmarks in the order shown in the title** (`LANDMARKS`, from `wfield_local/face_rhythm_cams.py`). **Right-click** removes the last point. A landmark you cannot see: press **n** to skip it (saved as null). Same anatomical point on every animal -- that is all the registration needs.
3. Save (last cell) -> `register_inputs/landmarks.json`, merged with what is there (other sessions are kept). Then run the registration without `--stage`.

Suggested definitions: **eye** = centre of the eye; **nose** = tip of the nose; **mouth** = corner of the mouth (side views) / centre of the closed lips (cam1); **chin** = lowest point of the chin; **ear** = front base of the ear; **cheek_l / cheek_r** (cam1) = the face's widest point on the image left / right at the level of the mouth.

In [ ]:
%matplotlib widget
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

CAM = "cam2"                     # cam1 | cam2 | cam3
FIGSIZE = 10
ROOT = Path(r"M:\MICROSCOPE\Priya\DeepLabCut\Widefield") / "face_rhythm" / CAM / "register_inputs"
INFO = json.loads((ROOT / "click_sessions.json").read_text())
LANDMARKS = INFO["landmarks"]
SESSIONS = sorted(set(INFO["templates"].values()))
OUT = ROOT / "landmarks.json"
CLICKS = json.loads(OUT.read_text()) if OUT.exists() else {}
print("master", INFO["master"], "| sessions", SESSIONS, "| landmarks", LANDMARKS)

In [ ]:
def click(session):
    """Click LANDMARKS in order on the session's rest face; right-click = undo, 'n' = skip (null)."""
    import matplotlib.image as mpimg
    img = mpimg.imread(ROOT / f"click_{session}.png")
    got = {k: v for k, v in CLICKS.get(session, {}).items()}
    order = list(LANDMARKS)
    fig, ax = plt.subplots(figsize=(FIGSIZE, FIGSIZE * img.shape[0] / img.shape[1]))
    ax.imshow(img, cmap="gray"); ax.axis("off")
    arts = []

    def redraw():
        for a in arts:
            a.remove()
        arts.clear()
        for k, v in got.items():
            if v is not None:
                arts.append(ax.plot(*v, "+", ms=14, mew=2, color="yellow")[0])
                arts.append(ax.text(v[0] + 6, v[1] - 6, k, color="yellow", fontsize=10))
        nxt = [k for k in order if k not in got]
        ax.set_title(f"{session}: next = {nxt[0] if nxt else 'DONE'}   (order {order})", fontsize=10)
        fig.canvas.draw_idle()

    def on_click(ev):
        if ev.inaxes is not ax or fig.canvas.toolbar.mode:
            return
        nxt = [k for k in order if k not in got]
        if ev.button == 3 and got:
            got.pop(list(got)[-1])
        elif ev.button == 1 and nxt:
            got[nxt[0]] = [round(float(ev.xdata), 1), round(float(ev.ydata), 1)]
        CLICKS[session] = dict(got); redraw()

    def on_key(ev):
        nxt = [k for k in order if k not in got]
        if ev.key == "n" and nxt:
            got[nxt[0]] = None; CLICKS[session] = dict(got); redraw()

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    redraw(); plt.show()

In [ ]:
click(SESSIONS[0])

In [ ]:
click(SESSIONS[1])

In [ ]:
click(SESSIONS[2])

In [ ]:
click(SESSIONS[3])

In [ ]:
# save: merged with the file's other sessions; complete = every landmark clicked or skipped
assert "MICROSCOPE" in str(OUT) and "Priya" in str(OUT)
old = json.loads(OUT.read_text()) if OUT.exists() else {}
old.update(CLICKS)
OUT.write_text(json.dumps(old, indent=1))
for s in SESSIONS:
    print(s, "complete" if all(k in old.get(s, {}) for k in LANDMARKS) else "INCOMPLETE", old.get(s))